In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
import findspark

findspark.init("/Users/i545672/spark3/spark-3.5.5-bin-hadoop3")
findspark.find()

'/Users/i545672/spark3/spark-3.5.5-bin-hadoop3'

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = (
    SparkSession
        .builder
        .appName("SparkSQLApp")
        .master("local[4]")
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.sql.adaptive.enabled", "false")
        .getOrCreate()
)

sc= spark.sparkContext
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/24 21:35:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
taxiSchema = StructType([
 StructField("VendorID", IntegerType(), True),
 StructField("tpep_pickup_datetime", TimestampType(), True),
 StructField("tpep_dropoff_datetime", TimestampType(), True),
 StructField("passenger_count", DoubleType(), True),
 StructField("trip_distance", DoubleType(), True),
 StructField("RatecodeID", DoubleType(), True),
 StructField("store_and_fwd_flag", StringType(), True),
 StructField("PULocationID", IntegerType(), True),
 StructField("DOLocationID", IntegerType(), True),
 StructField("payment_type", IntegerType(), True),
 StructField("fare_amount", DoubleType(), True),
 StructField("extra", DoubleType(), True),
 StructField("mta_tax", DoubleType(), True),
 StructField("tip_amount", DoubleType(), True),
 StructField("tolls_amount", DoubleType(), True),
 StructField("improvement_surcharge", DoubleType(), True),
 StructField("total_amount", DoubleType(), True),
 StructField("congestion_surcharge", DoubleType(), True),
 StructField("airport_fee", DoubleType(), True),
])

In [5]:
yellowTaxisDF = spark.read.option("header", "true").schema(taxiSchema).csv(
    "./Files/YellowTaxis_202210.csv"
)
yellowTaxisDF.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [6]:
yellowTaxisDF.createOrReplaceTempView("YellowTaxis")

In [7]:
outputDF = spark.sql("SELECT * FROM YellowTaxis WHERE PULocationID = 171")
outputDF.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|1       |2022-10-01 13:47:23 |2022-10-01 14:38:50  |1.0            |9.4          |99.0      |N                 |171         |263         |1           |35.2       |0.0  |0.5    |0.0      

25/05/24 21:36:05 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [8]:
greenTaxisDF = spark.read.option("header", "true").option("delimiter", "\t").csv(
    "./Files/GreenTaxis_202210.csv"
)
greenTaxisDF.createOrReplaceTempView("GreenTaxis")
greenTaxisDF.printSchema()

root
 |-- VendorId: string (nullable = true)
 |-- lpep_pickup_datetime: string (nullable = true)
 |-- lpep_dropoff_datetime: string (nullable = true)
 |-- passenger_count: string (nullable = true)
 |-- trip_distance: string (nullable = true)
 |-- RatecodeID: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: string (nullable = true)
 |-- DOLocationID: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: string (nullable = true)
 |-- extra: string (nullable = true)
 |-- mta_tax: string (nullable = true)
 |-- tip_amount: string (nullable = true)
 |-- tolls_amount: string (nullable = true)
 |-- improvement_surcharge: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- congestion_surcharge: string (nullable = true)
 |-- airport_fee: string (nullable = true)



In [11]:
# Union Yellow and Green Taxis
unionDF = spark.sql("""SELECT 'Yellow' as TaxiType,
        tpep_pickup_datetime AS PickupTime,
        tpep_dropoff_datetime AS DropoffTime,
        PULocationID AS PickupLocationID,
        DOLocationID AS DropoffLocationID
    FROM YellowTaxis
    UNION ALL
    SELECT 'Green' as TaxiType,
        lpep_pickup_datetime AS PickupTime,
        lpep_dropoff_datetime AS DropoffTime,
        PULocationID AS PickupLocationID,
        DOLocationID AS DropoffLocationID
    FROM GreenTaxis
""").show()

+--------+-------------------+-------------------+----------------+-----------------+
|TaxiType|         PickupTime|        DropoffTime|PickupLocationID|DropoffLocationID|
+--------+-------------------+-------------------+----------------+-----------------+
|  Yellow|2022-10-01 05:33:41|2022-10-01 05:48:39|             249|              107|
|  Yellow|2022-10-01 05:44:30|2022-10-01 05:49:48|             151|              238|
|  Yellow|2022-10-01 05:57:13|2022-10-01 06:07:41|             238|              166|
|  Yellow|2022-10-01 06:02:53|2022-10-01 06:08:55|             142|              239|
|  Yellow|2022-10-01 06:14:55|2022-10-01 06:20:21|             238|              166|
|  Yellow|2022-10-01 05:52:52|2022-10-01 06:22:14|             186|               41|
|  Yellow|2022-10-01 06:03:19|2022-10-01 06:14:51|             162|              145|
|  Yellow|2022-10-01 05:32:42|2022-10-01 06:20:01|             100|               22|
|  Yellow|2022-10-01 05:36:35|2022-10-01 05:54:38|    

In [15]:
taxiZonesSchema = 'LocationID INT, Borough STRING, Zone STRING, ServiceZone STRING'

taxiZonesDF = spark.read.schema(taxiZonesSchema).csv(
    "./Files/TaxiZones.csv"
)

taxiZonesDF.createOrReplaceGlobalTempView("TaxiZones")

taxiZonesDF.show()

+----------+-------------+--------------------+-----------+
|LocationID|      Borough|                Zone|ServiceZone|
+----------+-------------+--------------------+-----------+
|         1|          EWR|      Newark Airport|        EWR|
|         2|       Queens|         Jamaica Bay|  Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|  Boro Zone|
|         4|    Manhattan|       Alphabet City|Yellow Zone|
|         5|Staten Island|       Arden Heights|  Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|  Boro Zone|
|         7|       Queens|             Astoria|  Boro Zone|
|         8|       Queens|        Astoria Park|  Boro Zone|
|         9|       Queens|          Auburndale|  Boro Zone|
|        10|       Queens|        Baisley Park|  Boro Zone|
|        11|     Brooklyn|          Bath Beach|  Boro Zone|
|        12|    Manhattan|        Battery Park|Yellow Zone|
|        13|    Manhattan|   Battery Park City|Yellow Zone|
|        14|     Brooklyn|           Bay

In [ ]:
# Create a report to figure out number of rides grouped by borough and type of taxi.
# Merge all the taxi data using UNION ALL, and call it AllTaxis.
# Then join this dataset with TaxiZones
# Join them on pick‑up location column,
# Then group it by Borough and TaxiType, and count the number of trips.
# Finally, sort the data using Borough and TaxiType,
reportDF = spark.sql("""
    SELECT Borough, TaxiType, COUNT(*) AS TotalTrips
    FROM global_temp.TaxiZones
    LEFT JOIN (
        SELECT 'Yellow' as TaxiType, PULocationID FROM YellowTaxis
        UNION ALL
        SELECT 'Green' as TaxiType, PULocationID FROM GreenTaxis
    ) AllTaxis
    ON AllTaxis.PULocationID = TaxiZones.LocationID
    GROUP BY Borough, TaxiType
    ORDER BY Borough, TaxiType
""")
reportDF.show()

+-------------+--------+----------+
|      Borough|TaxiType|TotalTrips|
+-------------+--------+----------+
|        Bronx|   Green|      1852|
|        Bronx|  Yellow|      4511|
|     Brooklyn|   Green|     11113|
|     Brooklyn|  Yellow|     28089|
|          EWR|   Green|        15|
|          EWR|  Yellow|      1157|
|    Manhattan|    NULL|         2|
|    Manhattan|   Green|     40545|
|    Manhattan|  Yellow|   3250695|
|       Queens|    NULL|         1|
|       Queens|   Green|     15377|
|       Queens|  Yellow|    333922|
|Staten Island|    NULL|         2|
|Staten Island|   Green|         8|
|Staten Island|  Yellow|       303|
|      Unknown|   Green|       412|
|      Unknown|  Yellow|     56735|
+-------------+--------+----------+

